Team (5 BBA SBA B):
1. Niharika Chukkambotla (2423302)
2. Pournami Jayaraj (2423313)
3. Meghna Nikhil (2423317)
4. Sarthak Chaudhary (2423346)

In [1]:
import pandas as pd
import numpy as np
         
SNAPSHOT       = pd.Timestamp("2011-12-10")
GROSS_MARGIN   = 0.25
RETENTION_RATE = 0.70
DISCOUNT_RATE  = 0.10

TASK B1: CLEAN AND LOAD THE DATASET

In [2]:
df = pd.read_excel(r"Python Online Retail.xlsx")
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France


In [3]:
raw_rows = len(df)

In [4]:
clean = df.drop_duplicates()
clean = clean.dropna(subset=["CustomerID"])
clean = clean[~clean["InvoiceNo"].astype(str).str.startswith("C")]
clean = clean[(clean["Quantity"] > 0) & (clean["UnitPrice"] > 0)]

In [5]:
clean["CustomerID"] = clean["CustomerID"].astype(int)
clean["Revenue"] = clean["Quantity"] * clean["UnitPrice"]

print(f"B1 | raw={raw_rows} clean={len(clean)} dropped_pct={(raw_rows-len(clean))/raw_rows:.1%}")

B1 | raw=541909 clean=392692 dropped_pct=27.5%


Interpretation: 27.5% of raw records were invalid for customer-level analytics, primarily due to guest checkouts missing a CustomerID, order cancellations, and non-commercial test records.

TASK B2: CUSTOMER-LEVEL TABLE

In [6]:
clean["InvoiceDate"] = pd.to_datetime(clean["InvoiceDate"])
clean["DateOnly"] = clean["InvoiceDate"].dt.normalize()

cust = clean.groupby("CustomerID").agg(
    Orders=("InvoiceNo", "nunique"),
    Revenue=("Revenue", "sum"),
    FirstPurchase=("DateOnly", "min"),
    LastPurchase=("DateOnly", "max")
).reset_index()

cust["Recency"] = (SNAPSHOT - cust["LastPurchase"]).dt.days
cust["TenureDays"] = (cust["LastPurchase"] - cust["FirstPurchase"]).dt.days
cust["AOV"] = cust["Revenue"] / cust["Orders"]

print(f"B2 | customers={len(cust)} total_revenue={cust['Revenue'].sum():,.2f} "
      f"mean_orders={cust['Orders'].mean():.2f} median_orders={cust['Orders'].median():.0f}")

B2 | customers=4338 total_revenue=8,887,208.89 mean_orders=4.27 median_orders=2


Interpretation: The business has a solid base of 4,338 distinct customers generating £8.89M. Order volume is right-skewed: while the average is 4.27 orders per customer, the median is only 2, showing that heavy buyers pull up the average.

TASK B3: REPEAT PURCHASE BEHAVIOUR

In [7]:
repeat_rate = (cust["Orders"] >= 2).mean()
one_timers = (cust["Orders"] == 1).sum()
overall_aov = clean.groupby("InvoiceNo")["Revenue"].sum().mean()

print(f"B3 | repeat_rate={repeat_rate:.2%} one_timers={one_timers} overall_aov={overall_aov:.2f}")

B3 | repeat_rate=65.58% one_timers=1493 overall_aov=479.56


Interpretation: Nearly two-thirds of customers (65.58%) make repeat purchases, demonstrating strong product-market fit. However, the 1,493 single-order customers (34.42%) highlight a major drop-off after the first purchase that requires targeted post-purchase onboarding.

TASK B4: PREDICTIVE CLV

In [8]:
window_years = (clean["DateOnly"].max() - clean["DateOnly"].min()).days / 365.0
multiple = RETENTION_RATE / (1 + DISCOUNT_RATE - RETENTION_RATE)

cust["AnnualOrders"] = cust["Orders"] / window_years
cust["AnnualProfit"] = cust["AOV"] * cust["AnnualOrders"] * GROSS_MARGIN
cust["CLV"] = cust["AnnualProfit"] * multiple
cust["HistoricCLV"] = cust["Revenue"] * GROSS_MARGIN

print(f"B4 | window_years={window_years:.4f} multiple={multiple:.2f} "
      f"mean_CLV={cust['CLV'].mean():.2f} median_CLV={cust['CLV'].median():.2f} "
      f"total_CLV={cust['CLV'].sum():,.0f}")
print("B4 | top 5 by CLV:")
print(cust.nlargest(5, "CLV")[["CustomerID", "Orders", "AOV", "Revenue", "CLV"]]
        .round(2).to_string(index=False))

B4 | window_years=1.0219 multiple=1.75 mean_CLV=877.08 median_CLV=286.23 total_CLV=3,804,762
B4 | top 5 by CLV:
 CustomerID  Orders      AOV   Revenue       CLV
      14646      73  3838.44 280206.02 119960.85
      18102      60  4327.62 259657.30 111163.61
      17450      46  4225.89 194390.79  83221.93
      16446       2 84236.25 168472.50  72125.88
      14911     201   714.98 143711.17  61525.14


Interpretation:

Total Future Value: The entire customer base is projected to generate about £3.80 million in total future profit over time.

High Value Skew: The average customer value is £877.08, but the typical (median) customer is worth only £286.23. This large gap means a small group of high spenders is pulling up the overall average.

Wholesale Outlier Distortion: Customer 16446 has a massive projected CLV of £72.1k from just 2 wholesale orders. Because this bulk purchase inflates their average order value, the formula mistakenly assumes they will keep spending at that massive rate forever, which skews standard retail forecasts. 

TASK B5: VALUE CONCENTRATION

In [9]:
cust_sorted = cust.sort_values("CLV", ascending=False)
total_revenue = cust["Revenue"].sum()

for p in (0.01, 0.10, 0.20):
    n = int(np.ceil(p * len(cust)))
    share = cust_sorted.head(n)["Revenue"].sum() / total_revenue
    print(f"B5 | top {p:.0%} ({n} customers) hold {share:.1%} of revenue")

B5 | top 1% (44 customers) hold 32.1% of revenue
B5 | top 10% (434 customers) hold 61.5% of revenue
B5 | top 20% (868 customers) hold 74.7% of revenue


Interpretation: The customer base follows a classic Pareto distribution: the top 20% of buyers generate ~75% of total sales. Retention budgets and personalized account management must prioritize this top quintile to safeguard business revenue.  

TASK B6: RFM segments crossed with CLV

In [10]:
cust["R_score"] = pd.qcut(cust["Recency"], 5, labels=[5, 4, 3, 2, 1]).astype(int)
cust["F_score"] = pd.qcut(cust["Orders"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
cust["M_score"] = pd.qcut(cust["Revenue"], 5, labels=[1, 2, 3, 4, 5]).astype(int)

def segment_rfm(row):
    r, f = row["R_score"], row["F_score"]
    if r >= 4 and f >= 4:
        return "Champions"
    if r >= 3 and f >= 3:
        return "Loyal"
    if r >= 4 and f <= 2:
        return "New/Promising"
    if r <= 2 and f >= 4:
        return "At Risk"
    if r <= 2 and f <= 2:
        return "Lost"
    return "Needs Attention"

cust["Segment"] = cust.apply(segment_rfm, axis=1)

seg_table = cust.groupby("Segment").agg(
    Customers=("CustomerID", "count"),
    AvgCLV=("CLV", "mean"),
    TotalCLV=("CLV", "sum"),
    RevShare=("Revenue", "sum")
)
seg_table["RevShare%"] = (seg_table["RevShare"] / cust["Revenue"].sum()) * 100
seg_table = seg_table.drop(columns=["RevShare"]).sort_values("AvgCLV", ascending=False).round(2)

print(seg_table.to_string())
print("B6 | segment table printed above")

                 Customers   AvgCLV    TotalCLV  RevShare%
Segment                                                   
Champions             1128  2229.11  2514432.60      66.09
Loyal                  819   717.48   587613.41      15.44
At Risk                283   678.21   191932.38       5.04
Needs Attention        725   315.24   228548.63       6.01
Lost                  1071   208.76   223584.15       5.88
New/Promising          312   187.98    58650.67       1.54
B6 | segment table printed above


Interpretation: Champions and Loyal segments together account for 81.53% of total revenue, making them the core growth engine. Conversely, Lost accounts (1,071 customers) represent churned users with low average CLV (£208.76) where spending should be restricted to low-cost, automated re-engagement.

TASK B7: WRITTEN ANSWERS

**(a)** Customer **16446** (or **12346**) appears in the top-5 CLV list despite having very few orders due to an abnormally massive single purchase (wholesale/B2B behavior). Because this inflates AOV drastically without repeated buying sequence, the model projects extreme, unrealistic future lifetime value. We would **remove or separate them** from the B2C consumer model and evaluate them under a specialised B2B wholesale framework.

**(b)** Assuming a constant 70% retention rate treats all customers identically, regardless of whether they purchased yesterday or are long-churned one-time buyers, severely **overestimating the value of inactive/lost customers**. The alternative model to reach for is RFM Segmentation (or Segment-Based CLV), which separates customers by Recency and Frequency first so you can apply distinct retention rates and realistic valuations to active vs. lost customers.

INTERPRETATIONS:
1. Revenue is heavily skewed, with the top 10% of customers generating 61.5% of total revenue and the top 20% generating 74.7%
2. The "Champions" segment accounts for nearly two-thirds of total company revenue (66.09%) with the highest average CLV of EUR 2,229.11
3. While the repeat purchase rate is strong at 65.58%, 1,493 one-time buyers (34.42%) represent a critical drop-off requiring targeted second-order onboarding campaigns